# incheon_gyeonggi_area_clip

- 목적: 서울 외부 25km 경쟁권역에 필요한 경기·인천 격자만 추출
- 원칙: 서울 내부는 기존 서울 추정인구 테이블 사용, 외부권역만 별도 처리
- 주의: 500m 총량 보존을 위해 외부권역 후보가 속한 500m 격자 전체를 계산대상으로 확장


## 1. 경로 및 패키지

- 기존 `analysis_table` 경로 체계를 사용
- 산출물은 `analysis_table/data/output`에 저장


In [ ]:
import pathlib
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import pyogrio


BASE_PATH = pathlib.Path().resolve()

if BASE_PATH.name == "notebooks":
    BASE_PATH = BASE_PATH.parent
elif BASE_PATH.name != "analysis_table" and (BASE_PATH / "analysis_table").exists():
    BASE_PATH = BASE_PATH / "analysis_table"

PROJECT_PATH = BASE_PATH.parent
INPUT_PATH = BASE_PATH / "data" / "input"
OUTPUT_PATH = BASE_PATH / "data" / "output"
GRID_PATH = PROJECT_PATH / "data" / "grid"
BOUNDARY_PATH = PROJECT_PATH / "data" / "raw" / "spatial" / "boundary"
HOUSE_PATH = GRID_PATH / "주택수" / "2024_100M_all"

# 서울 외부 25km 분석권역에 실제 포함되는 경기·인천 원자료 단위만 사용
GYEONGGI_TARGET_UNITS = [
    "가평군", "고양시", "과천시", "광명시", "광주시", "구리시", "군포시", "김포시",
    "남양주시", "동두천시", "부천시", "성남시", "수원시", "시흥시", "안산시", "안양시",
    "양주시", "양평군", "여주시", "용인시", "의왕시", "의정부시", "파주시", "포천시",
    "하남시", "화성시"
]

INCHEON_TARGET_UNITS = [
    "강화군", "계양구", "남동구", "동구", "미추홀구", "부평구", "서구", "연수구", "중구"
]

REGION_TARGET_UNITS = {
    "경기": GYEONGGI_TARGET_UNITS,
    "인천": INCHEON_TARGET_UNITS
}

OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

print("BASE_PATH:", BASE_PATH)
print("GRID_PATH:", GRID_PATH)
print("OUTPUT_PATH:", OUTPUT_PATH)
print("HOUSE_PATH 존재:", HOUSE_PATH.exists())
print("경기 사용 시군 수:", len(GYEONGGI_TARGET_UNITS))
print("인천 사용 시군구 수:", len(INCHEON_TARGET_UNITS))

## 2. 서울 외부 25km 권역 생성

- 서울 행정동 경계를 하나로 통합
- 서울 경계 25km 버퍼에서 서울 내부를 제외
- 외부 경쟁 수요·공급 후보권역으로 사용


In [ ]:
seoul_bound = gpd.read_file(OUTPUT_PATH / "서울시_시군구_행정동_경계.gpkg")
print("서울 경계 CRS:", seoul_bound.crs)
print("서울 경계 구조:", seoul_bound.shape)

seoul_bound = seoul_bound.to_crs(5179)
seoul_union = seoul_bound.dissolve()
seoul_union["geometry"] = seoul_union.geometry.buffer(0)

outer_25km = seoul_union.copy()
outer_25km["geometry"] = seoul_union.geometry.buffer(25000).difference(seoul_union.geometry.iloc[0])
outer_25km = outer_25km[["geometry"]].copy()
outer_25km["분석권역"] = "서울외부25km"
outer_25km = gpd.GeoDataFrame(outer_25km, geometry="geometry", crs=5179)

print("외부 25km 권역 구조:", outer_25km.shape)
print("외부 25km 권역 면적(km2):", round(outer_25km.area.sum() / 1_000_000, 2))
print("geometry type:", outer_25km.geometry.geom_type.unique())

## 3. 행정동 경계 및 코드표

- 경기·인천 행정동 경계를 불러옴
- 100m 격자 중심점 기준으로 행정동을 결합할 때 사용


In [ ]:
adm = gpd.read_file(BOUNDARY_PATH / "BND_ADM_DONG_PG.shp")
print("행정동 경계 원본 CRS:", adm.crs)

adm = adm.to_crs(5179)
adm["행정동코드"] = adm["ADM_CD"].astype(str).str.zfill(8)

adm_code = pd.read_excel(
    BOUNDARY_PATH / "BND_ADM_DONG_PG_geocode.xlsx",
    sheet_name="2025년 6월",
    header=1,
    dtype=str
)

adm_code = adm_code.rename(columns={
    "시도명칭": "시도",
    "시군구명칭": "시군구",
    "읍면동명칭": "행정동"
})

adm_code["행정동코드"] = (
    adm_code["시도코드"].str.zfill(2)
    + adm_code["시군구코드"].str.zfill(3)
    + adm_code["읍면동코드"].str.zfill(3)
)

adm = adm.merge(
    adm_code[["행정동코드", "시도", "시군구", "행정동"]],
    on="행정동코드",
    how="left"
)

adm = adm[adm["시도"].isin(["경기도", "인천광역시"])].copy()
adm = adm[["행정동코드", "시도", "시군구", "행정동", "geometry"]].copy()

print("경기·인천 행정동 경계 구조:", adm.shape)
print("행정동코드 결측:", adm["행정동코드"].isna().sum())
print("시군구 결측:", adm["시군구"].isna().sum())
print(adm["시도"].value_counts())

## 4. 입력 격자 불러오기 함수

- 100m 총인구, 500m 총인구, 주거용도면적 shp를 동일한 방식으로 정리
- 결측값은 0 처리
- 경계에 걸친 중복 격자는 같은 GRID 기준으로 합산


In [ ]:
def read_grid_shp(region_name, folder_name, value_col, grid_col):
    folder_path = GRID_PATH / region_name / folder_name
    unit_list = REGION_TARGET_UNITS[region_name]

    shp_list = []
    missing_units = []

    for unit_name in unit_list:
        shp_path = folder_path / unit_name / "vl_blk.shp"
        if shp_path.exists():
            shp_list.append(shp_path)
        else:
            missing_units.append(unit_name)

    print(f"{region_name} {folder_name} 사용 단위 수:", len(unit_list))
    print(f"{region_name} {folder_name} shp 수:", len(shp_list))
    print(f"{region_name} {folder_name} 누락 단위:", missing_units)

    gdf_list = []

    for shp_path in shp_list:
        temp = gpd.read_file(shp_path, encoding="utf-8")
        temp = temp.rename(columns={"gid": grid_col, "val": value_col})
        temp[value_col] = pd.to_numeric(temp[value_col], errors="coerce").fillna(0)
        temp = temp[[grid_col, value_col, "geometry"]].copy()
        gdf_list.append(temp)

    if len(gdf_list) == 0:
        raise FileNotFoundError(f"{region_name} {folder_name}에서 읽을 shp가 없습니다.")

    result = pd.concat(gdf_list, ignore_index=True)
    result = gpd.GeoDataFrame(result, geometry="geometry", crs=gdf_list[0].crs)

    print("원본 구조:", result.shape)
    print("원본 GRID 중복:", result[grid_col].duplicated().sum())
    print("원본 값 결측:", result[value_col].isna().sum())

    result = (
        result
        .groupby(grid_col, as_index=False)
        .agg({
            value_col: "sum",
            "geometry": "first"
        })
    )
    result = gpd.GeoDataFrame(result, geometry="geometry", crs=gdf_list[0].crs)

    if result.crs != "EPSG:5179":
        result = result.to_crs(5179)

    print("중복 정리 후 구조:", result.shape)
    print("중복 정리 후 GRID 중복:", result[grid_col].duplicated().sum())

    return result


def read_region_grid(folder_name, value_col, grid_col):
    region_list = []

    for region_name in ["경기", "인천"]:
        temp = read_grid_shp(region_name, folder_name, value_col, grid_col)
        temp["원천시도"] = region_name
        region_list.append(temp)

    result = pd.concat(region_list, ignore_index=True)
    result = gpd.GeoDataFrame(result, geometry="geometry", crs=5179)

    print("=" * 100)
    print(f"경기·인천 {folder_name} 결합 구조:", result.shape)
    print("경기·인천 결합 GRID 중복:", result[grid_col].duplicated().sum())

    if result[grid_col].duplicated().sum() > 0:
        result = (
            result
            .groupby(grid_col, as_index=False)
            .agg({
                value_col: "sum",
                "geometry": "first"
            })
        )
        result = gpd.GeoDataFrame(result, geometry="geometry", crs=5179)
        print("경기·인천 재중복 정리 후 구조:", result.shape)
        print("경기·인천 재중복 정리 후 중복:", result[grid_col].duplicated().sum())

    return result

## 5. 경기·인천 100m·500m 격자 로딩

- 서울 외부 25km 분석권역에 포함되는 경기·인천 원자료 단위만 로딩
- 경기 사용 단위: 26개 시군
- 인천 사용 단위: 9개 군구
- 전체 경기·인천 원자료를 반복 로딩하지 않도록 선행 필터링

In [ ]:
pop100 = read_region_grid("100M총인구", "원본_인구수", "GRID_CD")
pop500 = read_region_grid("500M총인구", "총인구수_500", "GRID_CD_500")

pop100["중심점_x"] = pop100.geometry.centroid.x
pop100["중심점_y"] = pop100.geometry.centroid.y

pop100_point = gpd.GeoDataFrame(
    pop100[["GRID_CD", "중심점_x", "중심점_y"]].copy(),
    geometry=gpd.points_from_xy(pop100["중심점_x"], pop100["중심점_y"]),
    crs=5179
)

print("100m 전체 구조:", pop100.shape)
print("500m 전체 구조:", pop500.shape)
print("100m GRID 중복:", pop100["GRID_CD"].duplicated().sum())
print("500m GRID 중복:", pop500["GRID_CD_500"].duplicated().sum())

## 6. 외부 25km 후보 격자 추출

- 100m 격자 중심점이 서울 외부 25km 권역 안에 있는 격자를 후보로 선택
- 후보 격자가 속한 500m GRID를 확인


In [ ]:
candidate_join = gpd.sjoin(
    pop100_point[["GRID_CD", "geometry"]],
    outer_25km[["분석권역", "geometry"]],
    how="inner",
    predicate="within"
).drop(columns="index_right", errors="ignore")

candidate_grid = candidate_join[["GRID_CD"]].drop_duplicates().copy()

print("외부 25km 후보 100m 격자 수:", len(candidate_grid))
print("후보 GRID 중복:", candidate_grid["GRID_CD"].duplicated().sum())

candidate_point = pop100_point.merge(candidate_grid, on="GRID_CD", how="inner")

candidate_500_join = gpd.sjoin(
    candidate_point[["GRID_CD", "geometry"]],
    pop500[["GRID_CD_500", "geometry"]],
    how="left",
    predicate="within"
).drop(columns="index_right", errors="ignore")

print("후보 100m-500m 결합 결측:", candidate_500_join["GRID_CD_500"].isna().sum())
print("후보 100m-500m 중복:", candidate_500_join[["GRID_CD", "GRID_CD_500"]].duplicated().sum())

candidate_500_ids = (
    candidate_500_join["GRID_CD_500"]
    .dropna()
    .drop_duplicates()
    .to_frame()
)

print("계산대상 500m 격자 수:", len(candidate_500_ids))

## 7. 500m 단위 계산대상 확장

- 후보 100m 격자가 속한 500m 격자를 계산대상으로 설정
- 해당 500m에 포함된 100m 격자는 모두 포함
- 최종 외부권역 여부는 별도 플래그로 보존


In [ ]:
pop100_500_join = gpd.sjoin(
    pop100_point[["GRID_CD", "geometry"]],
    pop500[["GRID_CD_500", "geometry"]],
    how="left",
    predicate="within"
).drop(columns="index_right", errors="ignore")

print("전체 100m-500m 결합 결측:", pop100_500_join["GRID_CD_500"].isna().sum())
print("전체 100m-500m 결합 중복:", pop100_500_join[["GRID_CD", "GRID_CD_500"]].duplicated().sum())

calc_grid_ids = (
    pop100_500_join
    .merge(candidate_500_ids, on="GRID_CD_500", how="inner")
    [["GRID_CD", "GRID_CD_500"]]
    .drop_duplicates()
)

print("500m 확장 후 계산대상 100m 격자 수:", len(calc_grid_ids))
print("계산대상 GRID 중복:", calc_grid_ids["GRID_CD"].duplicated().sum())

calc100 = pop100.merge(calc_grid_ids, on="GRID_CD", how="inner")
calc100["계산대상_500m확장여부"] = True
calc100["외부25km권역여부"] = calc100["GRID_CD"].isin(candidate_grid["GRID_CD"])

calc500 = pop500.merge(candidate_500_ids, on="GRID_CD_500", how="inner")

print("최종 계산대상 100m 구조:", calc100.shape)
print("최종 계산대상 500m 구조:", calc500.shape)
print("외부25km권역 내부 100m 수:", calc100["외부25km권역여부"].sum())
print("500m 확장으로 추가된 100m 수:", (~calc100["외부25km권역여부"]).sum())

## 8. 행정동 결합

- 계산대상 100m 격자 중심점 기준으로 행정동을 결합
- 결측은 최근접 행정동으로 보정


In [ ]:
calc100_point = gpd.GeoDataFrame(
    calc100[["GRID_CD", "중심점_x", "중심점_y"]].copy(),
    geometry=gpd.points_from_xy(calc100["중심점_x"], calc100["중심점_y"]),
    crs=5179
)

adm_join = gpd.sjoin(
    calc100_point[["GRID_CD", "geometry"]],
    adm[["행정동코드", "시도", "시군구", "행정동", "geometry"]],
    how="left",
    predicate="within"
).drop(columns="index_right", errors="ignore")

print("행정동 결합 후 결측:", adm_join["행정동코드"].isna().sum())

missing_idx = adm_join["행정동코드"].isna()

if missing_idx.sum() > 0:
    near = gpd.sjoin_nearest(
        calc100_point.loc[missing_idx, ["GRID_CD", "geometry"]],
        adm[["행정동코드", "시도", "시군구", "행정동", "geometry"]],
        how="left",
        distance_col="행정동_보정거리"
    ).drop(columns="index_right", errors="ignore")
    
    near = near.drop_duplicates("GRID_CD")
    near_attr = near.set_index("GRID_CD")[["행정동코드", "시도", "시군구", "행정동"]]
    
    for col in ["행정동코드", "시도", "시군구", "행정동"]:
        adm_join.loc[missing_idx, col] = adm_join.loc[missing_idx, "GRID_CD"].map(near_attr[col])
    
    print("최근접 행정동 보정 격자 수:", len(near))
    print("최근접 행정동 보정 최대거리:", near["행정동_보정거리"].max())
    print("최근접 보정 후 행정동 결측:", adm_join["행정동코드"].isna().sum())

adm_attr = adm_join[["GRID_CD", "행정동코드", "시도", "시군구", "행정동"]].drop_duplicates("GRID_CD")

calc100 = calc100.merge(adm_attr, on="GRID_CD", how="left")

print("행정동 결합 후 구조:", calc100.shape)
print("행정동 결합 후 GRID 중복:", calc100["GRID_CD"].duplicated().sum())
print("행정동 결측:")
print(calc100[["행정동코드", "시군구", "행정동"]].isna().sum())

## 9. 주거면적·주택수 결합

- 주거면적은 GRID 기준 합산
- 주택수는 2024년 100m CSV에서 계산대상 GRID만 필터링
- 결측은 서울 전처리와 동일하게 0 처리


In [ ]:
area100 = read_region_grid("주거용도면적", "주거면적", "GRID_CD")
area100 = area100[["GRID_CD", "주거면적"]].copy()

calc100 = calc100.merge(area100, on="GRID_CD", how="left")
calc100["주거면적"] = calc100["주거면적"].fillna(0)

target_grid_set = set(calc100["GRID_CD"])
house_list = []

for csv_path in sorted(HOUSE_PATH.glob("*.csv")):
    temp = pd.read_csv(
        csv_path,
        header=None,
        names=["연도", "GRID_CD", "통계코드", "주택수"],
        dtype={"GRID_CD": str},
        encoding="cp949"
    )
    temp["주택수"] = pd.to_numeric(temp["주택수"], errors="coerce").fillna(0)
    temp = temp[temp["GRID_CD"].isin(target_grid_set)].copy()
    
    if len(temp) > 0:
        house_list.append(temp[["GRID_CD", "주택수"]])

if len(house_list) > 0:
    house100 = pd.concat(house_list, ignore_index=True)
    house100 = house100.groupby("GRID_CD", as_index=False)["주택수"].sum()
else:
    house100 = pd.DataFrame(columns=["GRID_CD", "주택수"])

print("주택수 결합대상 구조:", house100.shape)
print("주택수 GRID 중복:", house100["GRID_CD"].duplicated().sum())

calc100 = calc100.merge(house100, on="GRID_CD", how="left")
calc100["주택수"] = calc100["주택수"].fillna(0)

print("주거면적 결측:", calc100["주거면적"].isna().sum())
print("주택수 결측:", calc100["주택수"].isna().sum())
print("주거면적 0 격자 수:", (calc100["주거면적"] == 0).sum())
print("주택수 0 격자 수:", (calc100["주택수"] == 0).sum())

## 10. 최종 칼럼 정리

- 기존 서울 추정인구 테이블과 칼럼명을 최대한 맞춤
- 아직 인구 추정 전 단계이므로 `추정_인구수`는 생성하지 않음
- 외부권역 여부와 500m 확장 여부는 검토용 플래그로 유지


In [ ]:
calc100_final = calc100[[
    "시도",
    "GRID_CD",
    "행정동코드",
    "시군구",
    "행정동",
    "중심점_x",
    "중심점_y",
    "원본_인구수",
    "주거면적",
    "주택수",
    "GRID_CD_500",
    "외부25km권역여부",
    "계산대상_500m확장여부",
    "geometry"
]].copy()

calc100_final = gpd.GeoDataFrame(calc100_final, geometry="geometry", crs=5179)

calc500_final = calc500[[
    "GRID_CD_500",
    "총인구수_500",
    "geometry"
]].copy()

calc500_final = gpd.GeoDataFrame(calc500_final, geometry="geometry", crs=5179)

cnt_candidate = (
    calc100_final
    .groupby("GRID_CD_500", as_index=False)["외부25km권역여부"]
    .sum()
    .rename(columns={"외부25km권역여부": "외부25km후보_100m수"})
)

cnt_calc = (
    calc100_final
    .groupby("GRID_CD_500", as_index=False)["GRID_CD"]
    .count()
    .rename(columns={"GRID_CD": "계산대상_100m수"})
)

calc500_final = calc500_final.merge(cnt_candidate, on="GRID_CD_500", how="left")
calc500_final = calc500_final.merge(cnt_calc, on="GRID_CD_500", how="left")

print("최종 100m 구조:", calc100_final.shape)
print("최종 500m 구조:", calc500_final.shape)
print("100m GRID 중복:", calc100_final["GRID_CD"].duplicated().sum())
print("500m GRID 중복:", calc500_final["GRID_CD_500"].duplicated().sum())
print("100m 결측:")
print(calc100_final.isna().sum())
print("시도별 100m 격자 수:")
print(calc100_final["시도"].value_counts())
print("외부25km권역여부:")
print(calc100_final["외부25km권역여부"].value_counts())

## 11. 산출물 저장

- 외부 25km 권역 polygon 저장
- 500m 총량 보존 계산을 위한 100m·500m 계산대상 저장


In [ ]:
area_output = OUTPUT_PATH / "인천경기_외부25km_분석권역.gpkg"
grid100_output = OUTPUT_PATH / "인천경기_외부25km_100m_계산대상.gpkg"
grid500_output = OUTPUT_PATH / "인천경기_외부25km_500m_계산대상.gpkg"

outer_25km.to_file(area_output, driver="GPKG")
calc100_final.to_file(grid100_output, driver="GPKG")
calc500_final.to_file(grid500_output, driver="GPKG")

print("저장 완료:", area_output)
print("저장 완료:", grid100_output)
print("저장 완료:", grid500_output)
print("파일 존재 확인:", area_output.exists(), grid100_output.exists(), grid500_output.exists())

## 12. 500m 총량 기반 인구 노이즈 보정

- 서울 총인구 추정과 동일한 `지수4` 방식 적용
- `지수4 = 0.3×인구수비중 + 0.5×주택수비중 + 0.2×주거면적비중`
- 500m 총인구와 100m 원본 인구 합의 잔차를 지수4 비중으로 재배분
- 지수합이 0인 500m 격자는 내부 100m 격자 수 기준 균등분배
- 정수화는 500m 단위에서 내림 후 소수점이 큰 격자부터 1명씩 추가 배분


In [ ]:
calc100_path = OUTPUT_PATH / "인천경기_외부25km_100m_계산대상.gpkg"
calc500_path = OUTPUT_PATH / "인천경기_외부25km_500m_계산대상.gpkg"

calc100 = gpd.read_file(calc100_path)
calc500 = gpd.read_file(calc500_path)

print("100m 계산대상 구조:", calc100.shape)
print("500m 계산대상 구조:", calc500.shape)
print("100m GRID 중복:", calc100["GRID_CD"].duplicated().sum())
print("500m GRID 중복:", calc500["GRID_CD_500"].duplicated().sum())

pop_est = calc100.merge(
    calc500[["GRID_CD_500", "총인구수_500"]],
    on="GRID_CD_500",
    how="left"
)

print("500m 총인구 결합 후 결측:", pop_est["총인구수_500"].isna().sum())

pop_est["500원본인구수"] = pop_est.groupby("GRID_CD_500")["원본_인구수"].transform("sum")
pop_est["500주택수"] = pop_est.groupby("GRID_CD_500")["주택수"].transform("sum")
pop_est["500주거면적"] = pop_est.groupby("GRID_CD_500")["주거면적"].transform("sum")
pop_est["500격자수"] = pop_est.groupby("GRID_CD_500")["GRID_CD"].transform("count")

pop_est["인구수비중"] = np.where(
    pop_est["500원본인구수"] > 0,
    pop_est["원본_인구수"] / pop_est["500원본인구수"],
    0
)

pop_est["주택수비중"] = np.where(
    pop_est["500주택수"] > 0,
    pop_est["주택수"] / pop_est["500주택수"],
    0
)

pop_est["주거면적비중"] = np.where(
    pop_est["500주거면적"] > 0,
    pop_est["주거면적"] / pop_est["500주거면적"],
    0
)

pop_est["지수4"] = (
    pop_est["인구수비중"] * 0.3
    + pop_est["주택수비중"] * 0.5
    + pop_est["주거면적비중"] * 0.2
)

pop_est["지수4합"] = pop_est.groupby("GRID_CD_500")["지수4"].transform("sum")
pop_est["균등지수"] = np.where(pop_est["500격자수"] > 0, 1 / pop_est["500격자수"], 0)

pop_est["정규화지수4"] = np.where(
    pop_est["지수4합"] > 0,
    pop_est["지수4"] / pop_est["지수4합"],
    pop_est["균등지수"]
)

pop_est["인구잔차"] = pop_est["총인구수_500"] - pop_est["500원본인구수"]
pop_est["추정_인구수_raw"] = (
    pop_est["원본_인구수"]
    + pop_est["인구잔차"] * pop_est["정규화지수4"]
)

pop_est["추정_인구수_raw"] = pop_est["추정_인구수_raw"].clip(lower=0)

print("지수4합 0인 100m 격자 수:", (pop_est["지수4합"] == 0).sum())
print("지수4합 0인 500m 격자 수:", pop_est.loc[pop_est["지수4합"] == 0, "GRID_CD_500"].nunique())
print("추정 raw 음수:", (pop_est["추정_인구수_raw"] < 0).sum())
print("인구잔차 요약:")
print(
    pop_est
    .groupby("GRID_CD_500", as_index=False)["인구잔차"]
    .first()["인구잔차"]
    .describe()
)

pop_est["추정_인구수_내림"] = np.floor(pop_est["추정_인구수_raw"]).astype(int)
pop_est["추정_인구수_소수점"] = pop_est["추정_인구수_raw"] - pop_est["추정_인구수_내림"]

dist_pop = (
    pop_est
    .groupby("GRID_CD_500", as_index=False)
    .agg({
        "총인구수_500": "first",
        "추정_인구수_내림": "sum"
    })
    .rename(columns={"추정_인구수_내림": "내림합계"})
)

dist_pop["목표정수"] = dist_pop["총인구수_500"].round().astype(int)
dist_pop["분배_인구수"] = (dist_pop["목표정수"] - dist_pop["내림합계"]).astype(int)

print("분배_인구수 요약:")
print(dist_pop["분배_인구수"].describe())
print("분배_인구수 음수 500m 격자 수:", (dist_pop["분배_인구수"] < 0).sum())

pop_est = pop_est.merge(
    dist_pop[["GRID_CD_500", "분배_인구수"]],
    on="GRID_CD_500",
    how="left"
)

pop_est["추가배분순위"] = (
    pop_est
    .sort_values(
        ["GRID_CD_500", "추정_인구수_소수점", "GRID_CD"],
        ascending=[True, False, True]
    )
    .groupby("GRID_CD_500")
    .cumcount() + 1
)

pop_est["추가배분"] = np.where(
    pop_est["추가배분순위"] <= pop_est["분배_인구수"],
    1,
    0
)

pop_est["추정_인구수"] = pop_est["추정_인구수_내림"] + pop_est["추가배분"]

check_500 = (
    pop_est
    .groupby("GRID_CD_500", as_index=False)
    .agg({
        "총인구수_500": "first",
        "추정_인구수": "sum"
    })
)

check_500["정수화오차"] = check_500["총인구수_500"].round().astype(int) - check_500["추정_인구수"]

print("500m 정수화 오차 합:", check_500["정수화오차"].abs().sum())
print("500m 정수화 오차 격자 수:", (check_500["정수화오차"] != 0).sum())

pop_final = pop_est[[
    "시도",
    "GRID_CD",
    "행정동코드",
    "시군구",
    "행정동",
    "중심점_x",
    "중심점_y",
    "원본_인구수",
    "주거면적",
    "주택수",
    "추정_인구수",
    "GRID_CD_500",
    "외부25km권역여부",
    "계산대상_500m확장여부",
    "geometry"
]].copy()

pop_final = gpd.GeoDataFrame(pop_final, geometry="geometry", crs=5179)

pop_final_external = pop_final[pop_final["외부25km권역여부"]].copy()

print("계산대상 추정인구 구조:", pop_final.shape)
print("최종 외부25km 추정인구 구조:", pop_final_external.shape)
print("최종 외부25km GRID 중복:", pop_final_external["GRID_CD"].duplicated().sum())
print("최종 외부25km 결측:")
print(pop_final_external.isna().sum())
print("최종 외부25km 추정인구 합:", pop_final_external["추정_인구수"].sum())
print("시도별 최종 외부25km 격자 수:")
print(pop_final_external["시도"].value_counts())

final_output = OUTPUT_PATH / "인천경기_외부25km_100m_추정인구.gpkg"
pop_final_external.to_file(final_output, driver="GPKG")

print("저장 완료:", final_output)
print("파일 존재 확인:", final_output.exists())

## 13. 총인구 추정 검증 결과
- 적용 파일: `인천경기_외부25km_100m_추정인구.gpkg`
- 100m 계산대상: 449,445개, 최종 외부 25km 격자: 443,058개
- 중복 GRID_CD: 0개
- 주요 키 및 추정 인구 결측: 0개
- 500m 총량 정수화 오차: 0개
- 음수 추정 인구: 0개
- `지수4합=0`인 500m 격자는 내부 100m 격자 수 기준 균등분배함
- 최종 외부 25km 추정 인구 합계: 14,203,293명
- 시도별 추정 인구: 경기도 11,293,846명, 인천광역시 2,909,447명


## 14. 문화누리대상자 총량 데이터 전처리
- 문화누리대상자 총량은 `기초생활수급자 + 차상위계층수급권자`로 정의
- 기초생활수급자는 급여종류별 CSV 대신 `기초생활보장 수급자 수.csv`의 2024년 시군구 총량 사용
- 차상위계층은 `차상위및한부모가족수급자현황` 2024년 12월 시군구 총량 사용
- 경기 일부 시군구는 `수원시 권선구`처럼 구 단위로 표기되어 있어 복지 총량 자료 기준에 맞춰 시 단위로 통합
- 외부 25km만 분석하므로 시군구 전체 대상자 수에 `외부25km 추정인구 / 시군구 전체 500m 인구` 비율을 곱해 외부권역 대상자 총량 산정


In [ ]:
WELFARE_SOURCE_PATH = PROJECT_PATH / "data" / "raw" / "demographics" / "welfare" / "source"

mnc_pop_path = OUTPUT_PATH / "인천경기_외부25km_100m_추정인구.gpkg"
mnc_output_path = OUTPUT_PATH / "인천경기_외부25km_100m_문화누리대상자_추정인구.gpkg"

grid_mnc = gpd.read_file(mnc_pop_path)

print("외부25km 추정인구 구조:", grid_mnc.shape)
print("GRID 중복:", grid_mnc["GRID_CD"].duplicated().sum())
print("추정_인구수 결측:", grid_mnc["추정_인구수"].isna().sum())


def make_mnc_unit(row):
    sigungu = str(row["시군구"]).strip()
    if row["시도"] == "경기도" and " " in sigungu:
        first = sigungu.split()[0]
        if first.endswith("시"):
            return first
    return sigungu


grid_mnc["문화누리배분단위"] = grid_mnc.apply(make_mnc_unit, axis=1)

print("배분단위 수")
print(grid_mnc.groupby("시도")["문화누리배분단위"].nunique())

# 기초생활보장 수급자 총량
basic = pd.read_csv(GRID_PATH / "기초생활보장 수급자 수.csv", encoding="utf-8-sig")
basic = basic[(basic["시점"] == 2024) & basic["시도"].astype(str).str.contains("인천|경기", na=False)].copy()
basic = basic[basic["시군구"].notna()].copy()
basic["시도"] = np.where(basic["시도"].astype(str).str.contains("인천"), "인천광역시", "경기도")
basic = basic.rename(columns={"시군구": "문화누리배분단위", "값": "기초생활수급자수"})
basic["기초생활수급자수"] = pd.to_numeric(basic["기초생활수급자수"], errors="coerce").fillna(0).astype(int)
basic = basic[["시도", "문화누리배분단위", "기초생활수급자수"]].copy()


def read_classed_2024(sheet_name, sido_name):
    classed_file = WELFARE_SOURCE_PATH / "한국사회보장정보원_차상위및한부모가족수급자현황_통계표_2024.xlsx"
    temp = pd.read_excel(classed_file, sheet_name=sheet_name, header=None)
    start_idx = temp.index[temp.iloc[:, 0].astype(str).str.contains("시군구별", na=False)][0]
    result = temp.iloc[start_idx:, [0, 1, 13]].copy()
    result.columns = ["구분", "문화누리배분단위", "차상위계층수급권자수"]
    result["문화누리배분단위"] = result["문화누리배분단위"].ffill()
    result = result[result["문화누리배분단위"].notna()].copy()
    result = result[result["문화누리배분단위"].astype(str).str.strip() != ""].copy()
    result = result[~result["문화누리배분단위"].astype(str).str.contains("시군구별|구분|목차|nan", na=False)].copy()
    result["차상위계층수급권자수"] = pd.to_numeric(result["차상위계층수급권자수"], errors="coerce")
    result = result.dropna(subset=["차상위계층수급권자수"]).copy()
    result["시도"] = sido_name
    result["차상위계층수급권자수"] = result["차상위계층수급권자수"].astype(int)
    return result[["시도", "문화누리배분단위", "차상위계층수급권자수"]]


classed = pd.concat([
    read_classed_2024("5", "인천광역시"),
    read_classed_2024("10", "경기도")
], ignore_index=True)

mnc_total = basic.merge(classed, on=["시도", "문화누리배분단위"], how="outer")
mnc_total[["기초생활수급자수", "차상위계층수급권자수"]] = mnc_total[["기초생활수급자수", "차상위계층수급권자수"]].fillna(0).astype(int)
mnc_total["문화누리대상자_시군구전체"] = mnc_total["기초생활수급자수"] + mnc_total["차상위계층수급권자수"]

print("복지총량 구조:", mnc_total.shape)
print("복지총량 결측:", mnc_total.isna().sum().sum())
print("복지총량 합:", mnc_total["문화누리대상자_시군구전체"].sum())


def read_full_pop500(region_name):
    folder_path = GRID_PATH / region_name / "500M총인구"
    unit_list = REGION_TARGET_UNITS[region_name]
    pop_list = []
    missing_units = []

    for unit_name in unit_list:
        shp_path = folder_path / unit_name / "vl_blk.shp"

        if not shp_path.exists():
            missing_units.append(unit_name)
            continue

        temp = pyogrio.read_dataframe(
            shp_path,
            columns=["gid", "val"],
            read_geometry=False,
            encoding="UTF-8"
        )
        temp["val"] = pd.to_numeric(temp["val"], errors="coerce").fillna(0)
        temp["시도"] = "경기도" if region_name == "경기" else "인천광역시"
        temp["문화누리배분단위"] = unit_name
        pop_list.append(temp[["시도", "문화누리배분단위", "val"]])

    print(f"{region_name} 500m 총인구 사용 단위 수:", len(unit_list))
    print(f"{region_name} 500m 총인구 누락 단위:", missing_units)

    result = pd.concat(pop_list, ignore_index=True)
    result = result.groupby(["시도", "문화누리배분단위"], as_index=False)["val"].sum()
    result = result.rename(columns={"val": "배분단위_전체500m인구"})
    return result


full_pop500 = pd.concat([
    read_full_pop500("경기"),
    read_full_pop500("인천")
], ignore_index=True)

external_pop = (
    grid_mnc
    .groupby(["시도", "문화누리배분단위"], as_index=False)["추정_인구수"]
    .sum()
    .rename(columns={"추정_인구수": "외부25km_추정인구"})
)

mnc_unit = mnc_total.merge(full_pop500, on=["시도", "문화누리배분단위"], how="inner")
mnc_unit = mnc_unit.merge(external_pop, on=["시도", "문화누리배분단위"], how="inner")

mnc_unit["외부25km_인구비율"] = np.where(
    mnc_unit["배분단위_전체500m인구"] > 0,
    mnc_unit["외부25km_추정인구"] / mnc_unit["배분단위_전체500m인구"],
    0
)

ratio_over_count = (mnc_unit["외부25km_인구비율"] > 1).sum()
mnc_unit["외부25km_인구비율"] = mnc_unit["외부25km_인구비율"].clip(upper=1)

mnc_unit["문화누리대상자"] = (
    mnc_unit["문화누리대상자_시군구전체"]
    * mnc_unit["외부25km_인구비율"]
).round().astype(int)

print("외부권역 대상자 총량 구조:", mnc_unit.shape)
print("외부권역 인구비율 > 1 보정 건수:", ratio_over_count)
print("외부권역 문화누리대상자 합:", mnc_unit["문화누리대상자"].sum())
print(mnc_unit.groupby("시도")[["기초생활수급자수", "차상위계층수급권자수", "문화누리대상자_시군구전체", "문화누리대상자"]].sum())


## 15. 문화누리대상자 100m 격자 배분
- 서울 문화누리대상자 추정과 동일하게 `추정_인구수`와 `공시지가`를 함께 사용
- 공시지가 중복은 `GRID_CD` 평균값으로 정리
- 공시지가 결측은 `500m 평균 → 배분단위 평균 → 전체 중앙값` 순서로 보정
- 최종 배분식은 `beta=1.0` 적용
- 정수화는 배분단위별 내림 후 소수점이 큰 격자부터 잔여 인원 배분

$$
점수_i = 추정인구_i 	imes \exp[-eta 	imes \log(1 + 공시지가_i / c)]
$$

$$
문화누리대상자\ 추정_i = 대상자_{g} 	imes rac{점수_i}{\sum_{j \in g} 점수_j}
$$


In [ ]:
# 공시지가 결합
needed_grid = set(grid_mnc["GRID_CD"].astype(str))
land_list = []
land_file_count = 0
land_missing_units = []

for region_name in ["경기", "인천"]:
    land_folder = GRID_PATH / region_name / "공시지가"

    for unit_name in REGION_TARGET_UNITS[region_name]:
        shp_path = land_folder / unit_name / "vl_blk.shp"

        if not shp_path.exists():
            land_missing_units.append((region_name, unit_name))
            continue

        land_file_count += 1
        temp = pyogrio.read_dataframe(
            shp_path,
            columns=["gid", "val"],
            read_geometry=False,
            encoding="UTF-8"
        )
        temp = temp.rename(columns={"gid": "GRID_CD", "val": "공시지가"})
        temp = temp[temp["GRID_CD"].astype(str).isin(needed_grid)].copy()

        if len(temp) == 0:
            continue

        temp["공시지가"] = pd.to_numeric(temp["공시지가"], errors="coerce")
        land_list.append(temp[["GRID_CD", "공시지가"]])

land = pd.concat(land_list, ignore_index=True)
land_clean = land.groupby("GRID_CD", as_index=False)["공시지가"].mean()

print("공시지가 파일 수:", land_file_count)
print("공시지가 누락 단위:", land_missing_units)
print("공시지가 필터 후 구조:", land_clean.shape)
print("공시지가 GRID 중복:", land_clean["GRID_CD"].duplicated().sum())

grid_mnc = grid_mnc.merge(land_clean, on="GRID_CD", how="left", validate="one_to_one")
print("공시지가 결합 후 결측:", grid_mnc["공시지가"].isna().sum())

# 공시지가 결측 보정
grid_mnc["공시지가"] = grid_mnc["공시지가"].fillna(grid_mnc.groupby("GRID_CD_500")["공시지가"].transform("mean"))
print("500m 평균 보정 후 공시지가 결측:", grid_mnc["공시지가"].isna().sum())

grid_mnc["공시지가"] = grid_mnc["공시지가"].fillna(grid_mnc.groupby(["시도", "문화누리배분단위"])["공시지가"].transform("mean"))
print("배분단위 평균 보정 후 공시지가 결측:", grid_mnc["공시지가"].isna().sum())

land_median = grid_mnc.loc[grid_mnc["공시지가"] > 0, "공시지가"].median()
grid_mnc["공시지가"] = grid_mnc["공시지가"].fillna(land_median)
print("전체 중앙값 보정 후 공시지가 결측:", grid_mnc["공시지가"].isna().sum())

# 문화누리대상자 총량 결합
grid_mnc = grid_mnc.merge(
    mnc_unit[["시도", "문화누리배분단위", "문화누리대상자"]],
    on=["시도", "문화누리배분단위"],
    how="left",
    validate="many_to_one"
)

grid_mnc["문화누리대상자"] = grid_mnc["문화누리대상자"].fillna(0).astype(int)
print("문화누리대상자 총량 결합 결측:", grid_mnc["문화누리대상자"].isna().sum())

# beta=1.0 기준 배분
beta = 1.0
c = grid_mnc.loc[(grid_mnc["공시지가"] > 0) & (grid_mnc["추정_인구수"] > 0), "공시지가"].median()
print("상대 공시지가 기준 c:", c)

grid_mnc["로그상대공시지가"] = np.log1p(grid_mnc["공시지가"] / c)
grid_mnc["배분점수"] = np.exp(-beta * grid_mnc["로그상대공시지가"])
grid_mnc["격자별_배분점수"] = grid_mnc["추정_인구수"] * grid_mnc["배분점수"]
grid_mnc["배분단위_배분점수합"] = grid_mnc.groupby(["시도", "문화누리배분단위"])["격자별_배분점수"].transform("sum")
grid_mnc["배분단위_격자수"] = grid_mnc.groupby(["시도", "문화누리배분단위"])["GRID_CD"].transform("count")

grid_mnc["격자별_배분비율"] = np.where(
    grid_mnc["배분단위_배분점수합"] > 0,
    grid_mnc["격자별_배분점수"] / grid_mnc["배분단위_배분점수합"],
    1 / grid_mnc["배분단위_격자수"]
)

ratio_check = grid_mnc.groupby(["시도", "문화누리배분단위"], as_index=False)["격자별_배분비율"].sum()
print("배분비율 합 1 미일치 단위:", (~np.isclose(ratio_check["격자별_배분비율"], 1)).sum())

# 정수화
grid_mnc["문화누리대상자_추정_raw"] = grid_mnc["문화누리대상자"] * grid_mnc["격자별_배분비율"]
grid_mnc["문화누리대상자_추정_내림"] = np.floor(grid_mnc["문화누리대상자_추정_raw"]).astype(int)
grid_mnc["문화누리대상자_추정_소수점"] = grid_mnc["문화누리대상자_추정_raw"] - grid_mnc["문화누리대상자_추정_내림"]
grid_mnc["배분단위_내림합"] = grid_mnc.groupby(["시도", "문화누리배분단위"])["문화누리대상자_추정_내림"].transform("sum")
grid_mnc["배분단위_잔여배분인원"] = (grid_mnc["문화누리대상자"] - grid_mnc["배분단위_내림합"]).astype(int)

grid_mnc = grid_mnc.sort_values(
    ["시도", "문화누리배분단위", "문화누리대상자_추정_소수점", "GRID_CD"],
    ascending=[True, True, False, True]
).copy()

grid_mnc["배분단위_소수점순위"] = grid_mnc.groupby(["시도", "문화누리배분단위"]).cumcount() + 1

grid_mnc["문화누리대상자_추정_인구수"] = (
    grid_mnc["문화누리대상자_추정_내림"]
    + (grid_mnc["배분단위_소수점순위"] <= grid_mnc["배분단위_잔여배분인원"]).astype(int)
)

grid_mnc = grid_mnc.sort_values("GRID_CD").copy()

mnc_check = grid_mnc.groupby(["시도", "문화누리배분단위"], as_index=False).agg(
    목표문화누리대상자=("문화누리대상자", "first"),
    추정문화누리대상자=("문화누리대상자_추정_인구수", "sum")
)
mnc_check["정수화오차"] = mnc_check["목표문화누리대상자"] - mnc_check["추정문화누리대상자"]

print("정수화 오차 단위 수:", (mnc_check["정수화오차"] != 0).sum())
print("정수화 오차 절대합:", mnc_check["정수화오차"].abs().sum())
print("음수 문화누리 추정:", (grid_mnc["문화누리대상자_추정_인구수"] < 0).sum())
print("문화누리 추정인구 합:", grid_mnc["문화누리대상자_추정_인구수"].sum())
print(grid_mnc.groupby("시도").agg(
    격자수=("GRID_CD", "count"),
    추정인구=("추정_인구수", "sum"),
    문화누리대상자_추정인구=("문화누리대상자_추정_인구수", "sum")
))


## 16. 문화누리대상자 산출물 저장 및 검증 결과
- 최종 산출물: `인천경기_외부25km_100m_문화누리대상자_추정인구.gpkg`
- 최종 격자 수: 443,058개
- 최종 결측: 0개
- 정수화 오차: 0개
- 음수 추정값: 0개
- 문화누리대상자 추정 합계: 809,517명
- 시도별 추정 합계: 경기도 572,201명, 인천광역시 237,316명
- 특이사항: 외부권역 인구비율이 1을 초과한 9개 배분단위는 전체 시군구보다 외부권역 인구가 커질 수 없으므로 1로 상한 처리함
- 특이사항: 기초생활 급여종류별 파일은 중복 가능성이 있어 사용하지 않고, 시군구 총량 통계를 사용함


In [ ]:
final_cols = [
    "시도", "GRID_CD", "행정동코드", "시군구", "행정동", "문화누리배분단위",
    "중심점_x", "중심점_y", "원본_인구수", "주거면적", "주택수", "공시지가",
    "추정_인구수", "문화누리대상자_추정_인구수", "GRID_CD_500",
    "외부25km권역여부", "계산대상_500m확장여부", "geometry"
]

grid_mnc_final = grid_mnc[final_cols].copy()

if mnc_output_path.exists():
    mnc_output_path.unlink()

pyogrio.write_dataframe(grid_mnc_final, mnc_output_path, driver="GPKG")

print("저장 완료:", mnc_output_path)
print("최종 구조:", grid_mnc_final.shape)
print("최종 결측:")
print(grid_mnc_final.drop(columns="geometry").isna().sum())
print("최종 GRID 중복:", grid_mnc_final["GRID_CD"].duplicated().sum())
print("문화누리대상자 추정 합:", grid_mnc_final["문화누리대상자_추정_인구수"].sum())


## 17. 성별·연령별 인구자료 로딩

- 경기·인천 100m 성별·연령별 인구자료를 사용
- geometry는 사용하지 않고 `gid`, `val` 속성만 로딩
- 25km 외부권역 격자에 해당하는 GRID만 필터링
- 대형 shp 반복 로딩을 피하기 위해 DBF 속성표를 직접 읽음

In [ ]:
import struct
import warnings

warnings.filterwarnings("ignore", category=RuntimeWarning)


def read_dbf_gid_val(dbf_path, target_set, value_name):
    with open(dbf_path, "rb") as f:
        data = f.read()

    n_records = struct.unpack("<I", data[4:8])[0]
    header_len = struct.unpack("<H", data[8:10])[0]
    record_len = struct.unpack("<H", data[10:12])[0]

    fields = []
    pos = 32
    offset = 1
    while data[pos] != 0x0D:
        raw_name = data[pos:pos + 11].split(b"\x00", 1)[0]
        name = raw_name.decode("ascii", errors="ignore").lower()
        field_type = chr(data[pos + 11])
        field_len = data[pos + 16]
        fields.append((name, field_type, field_len, offset))
        offset += field_len
        pos += 32

    fmap = {name: (typ, length, off) for name, typ, length, off in fields}
    _, gid_len, gid_off = fmap["gid"]
    _, val_len, val_off = fmap["val"]

    gids = []
    vals = []
    base = header_len

    for i in range(n_records):
        rec = base + i * record_len
        if data[rec] == 0x2A:
            continue

        gid = data[rec + gid_off:rec + gid_off + gid_len].strip().decode("utf-8", errors="ignore")
        if gid not in target_set:
            continue

        raw_val = data[rec + val_off:rec + val_off + val_len].strip()
        if raw_val == b"":
            val = 0.0
        else:
            try:
                val = float(raw_val)
            except Exception:
                val = 0.0

        gids.append(gid)
        vals.append(val)

    return pd.DataFrame({"GRID_CD": gids, value_name: vals})

In [ ]:
mnc_path = OUTPUT_PATH / "인천경기_외부25km_100m_문화누리대상자_추정인구.gpkg"

base_cols = [
    "시도", "GRID_CD", "행정동코드", "시군구", "행정동", "문화누리배분단위",
    "추정_인구수", "문화누리대상자_추정_인구수", "GRID_CD_500", "외부25km권역여부"
]

grid = pyogrio.read_dataframe(mnc_path, read_geometry=False, columns=base_cols)
grid["추정_인구수"] = pd.to_numeric(grid["추정_인구수"], errors="coerce").fillna(0).round().astype("int64")
grid["문화누리대상자_추정_인구수"] = pd.to_numeric(grid["문화누리대상자_추정_인구수"], errors="coerce").fillna(0).round().astype("int64")
grid["비문화누리대상자_추정_인구수"] = (
    grid["추정_인구수"] - grid["문화누리대상자_추정_인구수"]
).clip(lower=0).astype("int64")

print("기본 격자 구조:", grid.shape)
print("GRID_CD 중복:", grid["GRID_CD"].duplicated().sum())
print("결측치 합:", grid.isna().sum().sum())
print("총 추정인구:", grid["추정_인구수"].sum())
print("문화누리대상자:", grid["문화누리대상자_추정_인구수"].sum())
print("비문화누리대상자:", grid["비문화누리대상자_추정_인구수"].sum())

## 18. 성별·연령별 원자료 정리

- 원자료의 성별 구분은 남성·여성 사용
- `총인구`, `유아인구`, `유소년인구`, `20대~100세 이상`을 로딩
- `0-5세`, `6-14세`, `15-19세`는 서울과 동일한 방식으로 파생
- 음수 파생 연령대는 발생하지 않음

In [ ]:
target_grid = set(grid["GRID_CD"].astype(str))
target_grid_df = pd.DataFrame({"GRID_CD": sorted(target_grid)})

region_paths = {
    "경기도": (GRID_PATH / "경기" / "격자100m_성연령별인구_2024_10", GYEONGGI_TARGET_UNITS),
    "인천광역시": (GRID_PATH / "인천" / "격자100m_성연령별인구_2024_10", INCHEON_TARGET_UNITS),
}

source_age_cols = [
    "총인구", "유아인구", "유소년인구",
    "20대인구", "30대인구", "40대인구", "50대인구",
    "60대인구", "70대인구", "80대인구", "90대인구", "100세이상인구"
]
sex_list = ["남성", "여성"]

wide_list = []

for sex in sex_list:
    sex_wide = target_grid_df.copy()

    for age in source_age_cols:
        part_list = []
        file_count = 0
        filtered_rows = 0

        missing_units = []

        for sido, (root, unit_list) in region_paths.items():
            for unit_name in unit_list:
                dbf_path = root / age / sex / unit_name / "vl_blk.dbf"

                if not dbf_path.exists():
                    missing_units.append((sido, unit_name))
                    continue

                file_count += 1
                temp = read_dbf_gid_val(dbf_path, target_grid, age)

                if len(temp) > 0:
                    part_list.append(temp)
                    filtered_rows += len(temp)

        if part_list:
            age_df = pd.concat(part_list, ignore_index=True).groupby("GRID_CD", as_index=False)[age].sum()
        else:
            age_df = pd.DataFrame({"GRID_CD": [], age: []})

        sex_wide = sex_wide.merge(age_df, on="GRID_CD", how="left")
        sex_wide[age] = pd.to_numeric(sex_wide[age], errors="coerce").fillna(0)

        print(f"{sex} {age}: dbf {file_count}, rows {filtered_rows}, missing {len(missing_units)}")

    sex_wide["성별"] = sex
    wide_list.append(sex_wide)

grid_stat_wide = pd.concat(wide_list, ignore_index=True)

age_over_20_cols = [
    "20대인구", "30대인구", "40대인구", "50대인구",
    "60대인구", "70대인구", "80대인구", "90대인구", "100세이상인구"
]

grid_stat_wide["0-19세"] = grid_stat_wide["총인구"] - grid_stat_wide[age_over_20_cols].sum(axis=1)
grid_stat_wide["0-5세"] = grid_stat_wide["유아인구"]
grid_stat_wide["6-14세"] = grid_stat_wide["유소년인구"] - grid_stat_wide["0-5세"]
grid_stat_wide["15-19세"] = grid_stat_wide["0-19세"] - grid_stat_wide["유소년인구"]

print("0-19세 음수:", (grid_stat_wide["0-19세"] < 0).sum())
print("6-14세 음수:", (grid_stat_wide["6-14세"] < 0).sum())
print("15-19세 음수:", (grid_stat_wide["15-19세"] < 0).sum())

grid_stat_wide = grid_stat_wide.rename(columns={
    "20대인구": "20-29세",
    "30대인구": "30-39세",
    "40대인구": "40-49세",
    "50대인구": "50-59세",
    "60대인구": "60-69세",
    "70대인구": "70-79세",
    "80대인구": "80-89세",
    "90대인구": "90-99세",
    "100세이상인구": "100세이상"
})

## 19. 성별·연령별 인구비중 생성

- wide 형태의 연령대 칼럼을 long 형태로 변환
- 격자별 전연령 인구수 합을 기준으로 성별·연령별 비중 산출
- 동일 GRID가 복수 시군구 자료에 반복되는 경우 GRID 기준으로 합산
- 성연령 원자료 비중이 없는 인구 격자는 행정동 비중 → 배분단위 비중 → 전체 비중 순으로 대체

In [ ]:
age_cols = [
    "0-5세", "6-14세", "15-19세",
    "20-29세", "30-39세", "40-49세", "50-59세",
    "60-69세", "70-79세", "80-89세", "90-99세", "100세이상"
]

grid_age_gender = grid_stat_wide[["GRID_CD", "성별"] + age_cols].melt(
    id_vars=["GRID_CD", "성별"],
    value_vars=age_cols,
    var_name="연령대",
    value_name="성연령격자_인구수"
)

grid_age_gender["성연령격자_인구수"] = pd.to_numeric(
    grid_age_gender["성연령격자_인구수"],
    errors="coerce"
).fillna(0).clip(lower=0)

grid_age_gender = grid_age_gender.groupby(
    ["GRID_CD", "성별", "연령대"],
    as_index=False
)["성연령격자_인구수"].sum()

grid_age_gender["격자별전연령인구수합"] = (
    grid_age_gender.groupby("GRID_CD")["성연령격자_인구수"].transform("sum")
)

grid_age_gender["격자별인구비중"] = np.where(
    grid_age_gender["격자별전연령인구수합"] > 0,
    grid_age_gender["성연령격자_인구수"] / grid_age_gender["격자별전연령인구수합"],
    0
)

ratio_check = (
    grid_age_gender
    .groupby("GRID_CD", as_index=False)["격자별인구비중"]
    .sum()
    .rename(columns={"격자별인구비중": "격자별인구비중합"})
)

print("격자별 성연령 비중 검토")
print(ratio_check["격자별인구비중합"].describe())
print("비중합 1 격자:", np.isclose(ratio_check["격자별인구비중합"], 1).sum())
print("비중합 0 격자:", np.isclose(ratio_check["격자별인구비중합"], 0).sum())

In [ ]:
base_grid = grid[[
    "시도", "GRID_CD", "행정동코드", "시군구", "행정동", "문화누리배분단위",
    "추정_인구수", "문화누리대상자_추정_인구수", "비문화누리대상자_추정_인구수"
]].copy()

grid_merge = base_grid.merge(grid_age_gender, on="GRID_CD", how="left")

print("병합 후 구조:", grid_merge.shape)
print("병합 후 결측치 합:", grid_merge.isna().sum().sum())
print("격자_성별_연령대 중복:", grid_merge[["GRID_CD", "성별", "연령대"]].duplicated().sum())

hjd_ratio = (
    grid_merge[grid_merge["격자별전연령인구수합"] > 0]
    .groupby(["시도", "시군구", "행정동", "성별", "연령대"], as_index=False)["성연령격자_인구수"]
    .sum()
)

hjd_ratio["행정동_전연령인구수합"] = (
    hjd_ratio.groupby(["시도", "시군구", "행정동"])["성연령격자_인구수"].transform("sum")
)

hjd_ratio["행정동_인구비중"] = np.where(
    hjd_ratio["행정동_전연령인구수합"] > 0,
    hjd_ratio["성연령격자_인구수"] / hjd_ratio["행정동_전연령인구수합"],
    0
)

hjd_ratio = hjd_ratio[["시도", "시군구", "행정동", "성별", "연령대", "행정동_인구비중"]].copy()

unit_ratio = (
    grid_merge[grid_merge["격자별전연령인구수합"] > 0]
    .groupby(["시도", "문화누리배분단위", "성별", "연령대"], as_index=False)["성연령격자_인구수"]
    .sum()
)

unit_ratio["배분단위_전연령인구수합"] = (
    unit_ratio.groupby(["시도", "문화누리배분단위"])["성연령격자_인구수"].transform("sum")
)

unit_ratio["배분단위_인구비중"] = np.where(
    unit_ratio["배분단위_전연령인구수합"] > 0,
    unit_ratio["성연령격자_인구수"] / unit_ratio["배분단위_전연령인구수합"],
    0
)

unit_ratio = unit_ratio[["시도", "문화누리배분단위", "성별", "연령대", "배분단위_인구비중"]].copy()

global_ratio = (
    grid_merge[grid_merge["격자별전연령인구수합"] > 0]
    .groupby(["성별", "연령대"], as_index=False)["성연령격자_인구수"]
    .sum()
)

global_total = global_ratio["성연령격자_인구수"].sum()
global_ratio["전체_인구비중"] = np.where(
    global_total > 0,
    global_ratio["성연령격자_인구수"] / global_total,
    0
)

global_ratio = global_ratio[["성별", "연령대", "전체_인구비중"]].copy()

grid_merge = grid_merge.merge(
    hjd_ratio,
    on=["시도", "시군구", "행정동", "성별", "연령대"],
    how="left"
)

grid_merge = grid_merge.merge(
    unit_ratio,
    on=["시도", "문화누리배분단위", "성별", "연령대"],
    how="left"
)

grid_merge = grid_merge.merge(
    global_ratio,
    on=["성별", "연령대"],
    how="left"
)

ratio_sum = (
    grid_merge
    .groupby("GRID_CD", as_index=False)
    .agg({
        "추정_인구수": "first",
        "문화누리대상자_추정_인구수": "first",
        "격자별인구비중": "sum"
    })
    .rename(columns={"격자별인구비중": "격자별인구비중합"})
)

under_grid = set(
    ratio_sum.loc[
        (ratio_sum["추정_인구수"] > 0) & (np.isclose(ratio_sum["격자별인구비중합"], 0)),
        "GRID_CD"
    ]
)

over_grid = set(
    ratio_sum.loc[
        (ratio_sum["추정_인구수"] == 0) & (~np.isclose(ratio_sum["격자별인구비중합"], 0)),
        "GRID_CD"
    ]
)

print("추정인구 0, 성연령 비중 있음:", len(over_grid))
print("추정인구 있음, 성연령 비중 0:", len(under_grid))

grid_merge["최종_인구비중"] = grid_merge["격자별인구비중"]
need_fallback = grid_merge["GRID_CD"].isin(under_grid)

grid_merge.loc[need_fallback, "최종_인구비중"] = grid_merge.loc[need_fallback, "행정동_인구비중"]

need_unit = need_fallback & (
    grid_merge["최종_인구비중"].isna() | np.isclose(grid_merge["최종_인구비중"], 0)
)
grid_merge.loc[need_unit, "최종_인구비중"] = grid_merge.loc[need_unit, "배분단위_인구비중"]

need_global = need_fallback & (
    grid_merge["최종_인구비중"].isna() | np.isclose(grid_merge["최종_인구비중"], 0)
)
grid_merge.loc[need_global, "최종_인구비중"] = grid_merge.loc[need_global, "전체_인구비중"]

grid_merge["최종_인구비중"] = grid_merge["최종_인구비중"].fillna(0)

# fallback 적용 후 격자별 비중합이 1이 되도록 재정규화
total_ratio = grid_merge.groupby("GRID_CD")["최종_인구비중"].transform("sum")
pos_idx = (grid_merge["추정_인구수"] > 0) & (total_ratio > 0)
grid_merge.loc[pos_idx, "최종_인구비중"] = grid_merge.loc[pos_idx, "최종_인구비중"] / total_ratio.loc[pos_idx]

final_ratio = (
    grid_merge
    .groupby("GRID_CD", as_index=False)
    .agg({
        "추정_인구수": "first",
        "최종_인구비중": "sum"
    })
    .rename(columns={"최종_인구비중": "최종_인구비중합"})
)

print("추정인구 있음, 최종비중 0 격자:", ((final_ratio["추정_인구수"] > 0) & np.isclose(final_ratio["최종_인구비중합"], 0)).sum())
print("추정인구 있음, 최종비중 1 아님 격자:", ((final_ratio["추정_인구수"] > 0) & ~np.isclose(final_ratio["최종_인구비중합"], 1)).sum())

## 20. 문화누리대상자·비문화누리대상자 성연령 배분

- 격자별 문화누리대상자 추정 인구수에 최종 성연령 비중을 곱함
- 일반 경쟁수요 반영을 위해 비문화누리대상자도 동일한 비중으로 배분
- 정수화는 격자별 내림 후 소수점이 큰 순서로 잔여 인원을 배분
- 최종 검증 결과 총인구, 문화누리대상자, 비문화누리대상자 모두 격자별 총량 오차 0

In [ ]:
grid_merge["문화누리대상자_성연령별_추정_인구수_raw"] = (
    grid_merge["문화누리대상자_추정_인구수"] * grid_merge["최종_인구비중"]
)

grid_merge["비문화누리대상자_성연령별_추정_인구수_raw"] = (
    grid_merge["비문화누리대상자_추정_인구수"] * grid_merge["최종_인구비중"]
)


def largest_remainder(df, target_col, raw_col, out_col):
    floor_col = out_col + "_floor"
    frac_col = out_col + "_소수"
    add_col = out_col + "_추가배분수"
    rank_col = out_col + "_추가배분순위"

    df[floor_col] = np.floor(df[raw_col]).astype("int32")
    df[frac_col] = df[raw_col] - df[floor_col]

    target = (
        df
        .groupby("GRID_CD", as_index=False)
        .agg({target_col: "first", floor_col: "sum"})
    )

    target["목표정수"] = target[target_col].round().astype("int64")
    target[add_col] = (target["목표정수"] - target[floor_col]).astype("int32")

    df = df.merge(target[["GRID_CD", add_col]], on="GRID_CD", how="left")

    df[rank_col] = (
        df
        .sort_values(["GRID_CD", frac_col], ascending=[True, False])
        .groupby("GRID_CD")
        .cumcount() + 1
    )

    df[out_col] = np.where(
        df[rank_col] <= df[add_col],
        1,
        0
    ).astype("int32") + df[floor_col]

    return df.drop(columns=[floor_col, frac_col, add_col, rank_col])


grid_merge = largest_remainder(
    grid_merge,
    "문화누리대상자_추정_인구수",
    "문화누리대상자_성연령별_추정_인구수_raw",
    "문화누리대상자_성연령별_추정_인구수"
)

grid_merge = largest_remainder(
    grid_merge,
    "비문화누리대상자_추정_인구수",
    "비문화누리대상자_성연령별_추정_인구수_raw",
    "비문화누리대상자_성연령별_추정_인구수"
)

grid_merge["추정_성연령별_인구수"] = (
    grid_merge["문화누리대상자_성연령별_추정_인구수"]
    + grid_merge["비문화누리대상자_성연령별_추정_인구수"]
).astype("int32")

final_cols = [
    "시도", "GRID_CD", "행정동코드", "시군구", "행정동", "문화누리배분단위",
    "성별", "연령대",
    "추정_성연령별_인구수",
    "문화누리대상자_성연령별_추정_인구수",
    "비문화누리대상자_성연령별_추정_인구수",
    "최종_인구비중"
]

external_age_gender = grid_merge[final_cols].copy()

zero_idx = (
    (external_age_gender["추정_성연령별_인구수"] == 0)
    & (external_age_gender["문화누리대상자_성연령별_추정_인구수"] == 0)
    & (external_age_gender["비문화누리대상자_성연령별_추정_인구수"] == 0)
)

external_age_gender_save = external_age_gender[~zero_idx].copy()

check = (
    external_age_gender_save
    .groupby("GRID_CD", as_index=False)
    .agg({
        "추정_성연령별_인구수": "sum",
        "문화누리대상자_성연령별_추정_인구수": "sum",
        "비문화누리대상자_성연령별_추정_인구수": "sum"
    })
)

check = grid[[
    "GRID_CD", "추정_인구수", "문화누리대상자_추정_인구수", "비문화누리대상자_추정_인구수"
]].merge(check, on="GRID_CD", how="left").fillna(0)

check["총인구_오차"] = check["추정_인구수"] - check["추정_성연령별_인구수"]
check["문화누리_오차"] = check["문화누리대상자_추정_인구수"] - check["문화누리대상자_성연령별_추정_인구수"]
check["비문화누리_오차"] = check["비문화누리대상자_추정_인구수"] - check["비문화누리대상자_성연령별_추정_인구수"]

print("전체 조합 구조:", external_age_gender.shape)
print("저장 조합 구조:", external_age_gender_save.shape)
print("저장 제외 all-zero 행:", zero_idx.sum())
print("최종 결측치 합:", external_age_gender_save.isna().sum().sum())
print("최종 중복:", external_age_gender_save[["GRID_CD", "성별", "연령대"]].duplicated().sum())
print("총인구 오차 격자:", (check["총인구_오차"] != 0).sum())
print("문화누리 오차 격자:", (check["문화누리_오차"] != 0).sum())
print("비문화누리 오차 격자:", (check["비문화누리_오차"] != 0).sum())

## 21. 성연령별 추정인구 산출물 저장

- 저장 파일: `인천경기_외부25km_100m_성연령별_추정인구.parquet`
- 저장 행 수: 778,205행
- all-zero 성별·연령대 조합은 저장 제외
- 총 추정인구: 14,203,293명
- 문화누리대상자 추정 인구수: 809,517명
- 비문화누리대상자 추정 인구수: 13,393,776명

In [ ]:
out_path = OUTPUT_PATH / "인천경기_외부25km_100m_성연령별_추정인구.parquet"

for col in ["시도", "시군구", "행정동", "문화누리배분단위", "성별", "연령대"]:
    external_age_gender_save[col] = external_age_gender_save[col].astype("category")

external_age_gender_save.to_parquet(
    out_path,
    index=False,
    compression="snappy"
)

print("저장 완료:", out_path)
print("저장 파일 크기 MB:", round(out_path.stat().st_size / 1024 / 1024, 2))

sido_summary = (
    external_age_gender_save
    .groupby("시도", observed=True)
    .agg({
        "추정_성연령별_인구수": "sum",
        "문화누리대상자_성연령별_추정_인구수": "sum",
        "비문화누리대상자_성연령별_추정_인구수": "sum",
        "GRID_CD": "nunique"
    })
    .reset_index()
)

display(sido_summary)

age_summary = (
    external_age_gender_save
    .groupby(["성별", "연령대"], observed=True)
    .agg({
        "추정_성연령별_인구수": "sum",
        "문화누리대상자_성연령별_추정_인구수": "sum"
    })
    .reset_index()
)

display(age_summary)